# 01 Audit dataset PhiUSIIL

Notebook ini menjalankan `src/audit.py` dan menampilkan ringkasannya. Semua angka disimpan di `results/audit.json`.

In [1]:
import os, sys, json
os.chdir('..')  # jalankan dari folder repo
sys.path.insert(0, '.')
import pandas as pd
from src import audit
audit.main.__module__

'src.audit'

In [2]:
import contextlib, io
with contextlib.redirect_stdout(io.StringIO()):
    audit.main()
a = json.load(open('results/audit.json', encoding='utf-8'))
{k: a[k] for k in ['file', 'shape', 'label_original', 'url_duplicates', 'after_dedup', 'filename_mw_prefix']}

{'file': {'name': 'PhiUSIIL_Phishing_URL_Dataset.csv',
  'bytes': 56854345,
  'sha256': 'a236549cd369cd80bd478ff8e1779cbf44c58d5c3f79f7a51a1adbed7d06d1c6'},
 'shape': {'rows': 235795,
  'columns': 56,
  'missing_values': 0,
  'infinite_values': 0,
  'full_row_duplicates': 0},
 'label_original': {'legitimate_1': 134850, 'phishing_0': 100945},
 'url_duplicates': {'urls_appearing_more_than_once': 425,
  'extra_rows_removed': 425,
  'urls_with_label_conflict': 0,
  'duplicate_rows_by_label': {'phishing': 425}},
 'after_dedup': {'rows': 235370, 'legitimate': 134850, 'phishing': 100520},
 'filename_mw_prefix': {'rows': 36286, 'phishing_among_them': 36286}}

## Templat URL sah per label

In [3]:
rows = []
for lab, d in a['template_by_label'].items():
    rows.append({'label': lab, 'n': d['rows'], **{k: f"{v['n']} ({v['pct']}%)" for k, v in d.items() if k != 'rows'}})
pd.DataFrame(rows).set_index('label')

,n,https,www,no_path_query,r0_match,sim100
label,,,,,,
legitimate,134850,134850 (100.0%),134850 (100.0%),134850 (100.0%),134850 (100.0%),134850 (100.0%)
phishing,100945,49196 (48.74%),41746 (41.36%),72272 (71.6%),1038 (1.03%),786 (0.78%)


## ROC-AUC satu fitur (10 teratas)

In [4]:
pd.DataFrame(a['single_feature_auc_top10'])

,feature,auc,oriented_auc
0,URLSimilarityIndex,0.0039,0.9961
1,LineOfCode,0.0097,0.9903
2,NoOfExternalRef,0.0120,0.9880
3,NoOfImage,0.0204,0.9796
4,NoOfSelfRef,0.0275,0.9725
5,NoOfJS,0.0292,0.9708
6,NoOfCSS,0.0424,0.9576
7,HasSocialNet,0.1053,0.8947
8,HasCopyrightInfo,0.1245,0.8755
9,HasDescription,0.1538,0.8462


## Hostname dan domain induk (eTLD+1)

In [5]:
print(a['hostname'])
print({k: v for k, v in a['registered_domain'].items() if k != 'top10'})
pd.DataFrame(a['registered_domain']['top10'])

{'unique': 220086, 'rows': 235795}
{'method': 'tldextract top_domain_under_public_suffix, ICANN PSL snapshot bawaan (offline)', 'unique': 175513, 'appearing_more_than_once': 6012, 'rows_in_those': 66294, 'mixed_label_domains': 120}


,domain,rows,phishing_pct
0,web.app,5754,100.0
1,firebaseapp.com,5594,100.0
2,repl.co,3754,100.0
3,weeblysite.com,3097,100.0
4,ipfs.io,1559,100.0
5,workers.dev,1438,100.0
6,square.site,1199,100.0
7,dweb.link,975,100.0
8,xsph.ru,928,100.0
9,pantheonsite.io,900,100.0
